# Load Libraries

In [1]:
!pip install -U "transformers>=4.42.3" bitsandbytes accelerate peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 569.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.32.1
    Uninstalling accelerate-0.32.1:
      Successfully uninstalled accelerate-0.32.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.3
    Uninstalling transformers-4.42.3:
      Successfully uninstalled transformers-4.42.3


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

VER=157

# FINAL SOLUTION IS USE_QLORA=FALSE, TRAIN_100_PERCENT=TRUE, ADD_33K=TRUE, DEBUG=FALSE
USE_QLORA = True
TRAIN_100_PERCENT = False
ADD_33K = False
DEBUG = True

In [3]:
import os
import copy
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    BitsAndBytesConfig,
    Gemma2ForSequenceClassification,
    GemmaTokenizerFast,
    Gemma2Config,
    PreTrainedTokenizerBase, 
    EvalPrediction,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import log_loss, accuracy_score

2024-08-14 21:21:22.008348: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-14 21:21:22.008473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-14 21:21:22.185976: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Configurations
Note that when HuggingFace uses `model parallelism` then total batch size equals `per_device_train_batch_size` times `gradient_accumulation_steps`. So the configuration below creates `total batch size = 8` **regardless of the number of GPUs**. 

When HuggingFace uses `data parallelism` then total batch size equals `per_device_train_batch_size` times `gradient_accumulation_steps` times `number of GPUs`.

If we load model with `device_map="auto"` then HuggingFace splits the model among all the GPUs and uses `model parallelism`. If we load the model onto a single GPU, then HuggingFace will make a copy of the model on each GPU and use `data parallelism`. More info about parallelism in my discussion post [here][1]

[1]: https://www.kaggle.com/competitions/lmsys-chatbot-arena/discussion/527596

In [4]:
@dataclass
class Config:
    output_dir: str = f"output-{VER}"
    checkpoint: str = "/kaggle/input/gemma2-9b-it-fp16"  
    max_length: int = 2048
    n_splits: int = 5
    fold_idx: int = 0
    optim_type: str = "adamw_8bit"
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4  # global batch size is 8 
    per_device_eval_batch_size: int = 4
    n_epochs: int = 1
    freeze_layers: int = 0 # there're 42 layers in total, we don't add adapters to the first 16 layers
    lr: float = 2e-4
    warmup_steps: int = 20
    lora_r: int = 64
    lora_alpha: float = 4 
    lora_dropout: float = 0.05
    lora_bias: str = "none"
    
config = Config()

# Training Arguments

In [5]:
training_args = TrainingArguments(
    output_dir = f"output-{VER}",
    overwrite_output_dir=True,
    report_to="none",
    num_train_epochs=config.n_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no", # don't save any checkpoints
    #save_steps=200,
    optim=config.optim_type,
    fp16=True, 
    #bf16=False,
    learning_rate=config.lr,
    warmup_steps=config.warmup_steps,

    #gradient_checkpointing=True, # this doesn't work correctly for some reason

    #logging_first_step=True,
    #lr_scheduler_type='linear', # "cosine" or "linear" or "constant" (default is linear)
    metric_for_best_model='log_loss',
    greater_is_better=False,  
    #save_total_limit=4,
    #load_best_model_at_end=True,
)

# LoRA config
We add 4 more target modules compared with the public notebook we forked.

In [6]:
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    # only target self-attention
    target_modules=["q_proj", "k_proj", "v_proj",
                    "down_proj","up_proj","o_proj","gate_proj"],
    layers_to_transform=[i for i in range(42) if i >= config.freeze_layers],
    lora_dropout=config.lora_dropout,
    bias=config.lora_bias,
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["score","classifier_head1", "classifier_head2"]
)

# Tokenizer and Custom 3-Head Model
From EDA we see there are 58 unique models in train data after renaming some. So we hardcode the value of 58 in code below.

In [7]:
tokenizer = GemmaTokenizerFast.from_pretrained(config.checkpoint)
tokenizer.add_eos_token = True  # We'll add <eos> at the end
tokenizer.padding_side = "right"

In [8]:
qlora = {}
if USE_QLORA:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4", #nf4 or fp4
        bnb_4bit_use_double_quant = False,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules = ["score","classifier_head1", "classifier_head2"]
    )
    qlora['quantization_config'] = bnb_config
    print("Using QLoRA")

Using QLoRA


In [9]:
import torch
import torch.nn as nn
from transformers import Gemma2ForSequenceClassification, Gemma2Config

class CustomGemma2ForSequenceClassification(Gemma2ForSequenceClassification):
    def __init__(self, config, num_labels_head1=58, num_labels_head2=58):
        super().__init__(config)
        self.num_labels_head1 = num_labels_head1
        self.num_labels_head2 = num_labels_head2
        self.classifier_head1 = nn.Linear(config.hidden_size, num_labels_head1, bias=False)
        self.classifier_head2 = nn.Linear(config.hidden_size, num_labels_head2, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device

        if labels is not None:
            labels = labels.to(device)
            outputs = super().forward(input_ids, attention_mask=attention_mask, labels=labels[:, 0], output_hidden_states=True)
        else:
            outputs = super().forward(input_ids, attention_mask=attention_mask)

        last_token_indices = (torch.sum(attention_mask, dim=1) - 1).to(device)
        last_token_outputs = outputs.hidden_states[-1].to(device)[
            torch.arange(outputs.hidden_states[-1].shape[0], device=device), last_token_indices]

        outputs_head1 = self.classifier_head1(last_token_outputs).to(device)
        outputs_head2 = self.classifier_head2(last_token_outputs).to(device)

        if labels is not None:
            labels_head1 = labels[:, 1].to(device)
            labels_head2 = labels[:, 2].to(device)
            
            loss_head1 = nn.CrossEntropyLoss()(outputs_head1, labels_head1)
            loss_head2 = nn.CrossEntropyLoss()(outputs_head2, labels_head2)
            loss = outputs.loss.to(device) + 0.1 * loss_head1 + 0.1 * loss_head2
            return {"loss": loss, "logits": (outputs.logits, outputs_head1, outputs_head2)}
        else:
            return {"logits": (outputs.logits, outputs_head1, outputs_head2)}

config2 = Gemma2Config.from_pretrained(config.checkpoint)
config2.num_labels = 3
model = CustomGemma2ForSequenceClassification.from_pretrained(
    config.checkpoint,
    config=config2,
    num_labels_head1=58,
    num_labels_head2=58,
    torch_dtype=torch.float16,
    device_map="auto",
    **qlora
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of CustomGemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/gemma2-9b-it-fp16 and are newly initialized: ['classifier_head1.weight', 'classifier_head2.weight', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): CustomGemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector):

In [10]:
model.print_trainable_parameters()

trainable params: 216,498,688 || all params: 9,458,631,168 || trainable%: 2.2889


# Load Training Data
We will train with 55k train data and 21k dedup data from [here][1]

[1]: https://www.kaggle.com/competitions/lmsys-chatbot-arena/discussion/500973

In [11]:
import pandas as pd

df = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv") 
df["id"] = df["id"].astype("str")
print('Competition data has shape', df.shape )
LN = len(df)
df.head(1)

Competition data has shape (57477, 9)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0


In [12]:
df.loc[df.model_a=='claude-1','model_a'] = 'claude-v1'
df.loc[df.model_b=='claude-1','model_b'] = 'claude-v1'

df.loc[df.model_a=='claude-instant-1','model_a'] = 'claude-instant-v1'
df.loc[df.model_b=='claude-instant-1','model_b'] = 'claude-instant-v1'

df.loc[df.model_a.str.contains("gpt-3.5"),'model_a'] = 'gpt-3.5-turbo'
df.loc[df.model_b.str.contains("gpt-3.5"),'model_b'] = 'gpt-3.5-turbo'

df.loc[df.model_a.str.contains("gpt-4"),'model_a'] = 'gpt-4'
df.loc[df.model_b.str.contains("gpt-4"),'model_b'] = 'gpt-4'

In [13]:
df2 = pd.read_csv("/kaggle/input/lmsys-additional-33k-labelled-conversations/lmsys-33k-deduplicated.csv")
print("External data has shape", df2.shape )
df2.head(1)

External data has shape (21187, 9)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,2564acd09e3942fd97657d05282d4389,oasst-pythia-12b,alpaca-13b,"[""Why did my parent not invite me to their wed...","[""It is possible that your parent did not invi...","[""It is likely that they wanted to keep the gu...",0,0,1


In [14]:
if ADD_33K:
    df = pd.concat([df,df2],axis=0,ignore_index=True)
if DEBUG:
    df = df.iloc[:64]
print("We will use train data with shape", df.shape )

We will use train data with shape (64, 9)


# Label Encode Models

In [15]:
import numpy as np
m1 = df.model_a.unique()
m2 = df.model_b.unique()
m = np.union1d(m1,m2)
m = sorted(m)
print(f"There are {len(m)} unique models:")

MAP = {x:y for x,y in zip(m,range(len(m)))}
print(MAP)

df.model_a = df.model_a.map(MAP).astype('int32')
df.model_b = df.model_b.map(MAP).astype('int32')
df.head(1)

There are 41 unique models:
{'alpaca-13b': 0, 'chatglm-6b': 1, 'chatglm2-6b': 2, 'chatglm3-6b': 3, 'claude-2.0': 4, 'claude-2.1': 5, 'claude-instant-v1': 6, 'claude-v1': 7, 'codellama-34b-instruct': 8, 'deepseek-llm-67b-chat': 9, 'dolly-v2-12b': 10, 'dolphin-2.2.1-mistral-7b': 11, 'gemini-pro': 12, 'gemini-pro-dev-api': 13, 'gpt-3.5-turbo': 14, 'gpt-4': 15, 'gpt4all-13b-snoozy': 16, 'guanaco-33b': 17, 'koala-13b': 18, 'llama-13b': 19, 'llama-2-13b-chat': 20, 'llama-2-70b-chat': 21, 'llama-2-7b-chat': 22, 'llama2-70b-steerlm-chat': 23, 'mistral-7b-instruct': 24, 'mistral-medium': 25, 'mixtral-8x7b-instruct-v0.1': 26, 'mpt-30b-chat': 27, 'oasst-pythia-12b': 28, 'openchat-3.5': 29, 'palm-2': 30, 'pplx-70b-online': 31, 'stablelm-tuned-alpha-7b': 32, 'starling-lm-7b-alpha': 33, 'tulu-2-dpo-70b': 34, 'vicuna-13b': 35, 'vicuna-33b': 36, 'vicuna-7b': 37, 'wizardlm-13b': 38, 'wizardlm-70b': 39, 'zephyr-7b-beta': 40}


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,15,15,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0


# Create Dataset

In [16]:
ds = Dataset.from_pandas(df)

In [17]:
import json

class CustomTokenizer:
    def __init__(
        self, 
        tokenizer: PreTrainedTokenizerBase, 
        max_length: int
    ) -> None:
        self.tokenizer = tokenizer
        self.max_length = max_length

    def prepare_text(self, prompts, responses_a, responses_b):

        prompts = json.loads(prompts)
        responses_a = json.loads(responses_a)
        responses_b = json.loads(responses_b)
        
        rounds = [
            f"<start_of_turn>prompt\n{prompts[i]}<end_of_turn>\n"
            +f"<start_of_turn>response_a\n{responses_a[i]}<end_of_turn>\n"
            +f"<start_of_turn>response_b\n{responses_b[i]}<end_of_turn>"
            for i in range(len(prompts))
        ]
        
        tmp = "\n".join(rounds)
        for k in range(len(rounds)):
            tmp = "\n".join(rounds[k:])
            if len( self.tokenizer(tmp)["input_ids"] ) < self.max_length: 
                break
        
        return tmp
        
    def __call__(self, batch: dict) -> dict:
        
        texts = [
            self.prepare_text(p, r_a, r_b)
            for p, r_a, r_b in zip(batch["prompt"], batch["response_a"], batch["response_b"])
        ]
        
        tokenized = self.tokenizer(texts, max_length=self.max_length, truncation=True)
        labels=[]
        for a_win, b_win, c, d in zip(batch["winner_model_a"], batch["winner_model_b"], 
                                   batch["model_a"],batch["model_b"]):
            if a_win:
                label = 0
            elif b_win:
                label = 1
            else:
                label = 2
            labels.append( (label,c,d) )
        return {**tokenized, "labels": labels} #, "texts": texts}

In [18]:
encode = CustomTokenizer(tokenizer, max_length=config.max_length)
ds = ds.map(encode, batched=True, num_proc=8)

/opt/conda/lib/python3.10/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Map (num_proc=8):   0%|          | 0/64 [00:00<?, ? examples/s]

# Compute Metrics

We'll compute the log-loss used in LB and accuracy as a auxiliary metric.

In [19]:
def compute_metrics(eval_preds: EvalPrediction) -> dict:
    preds = eval_preds.predictions
    labels = np.array( eval_preds.label_ids )
    
    # Split the predictions and labels into two heads
    preds_head1 = preds[0]
    preds_head2 = preds[1]
    preds_head3 = preds[2]
    labels_head1 = labels[:,0]
    labels_head2 = labels[:,1]
    labels_head3 = labels[:,2]
    
    # Compute log loss and accuracy for each head
    probs_head1 = torch.from_numpy(preds_head1).float().softmax(-1).numpy()
    loss_head1 = log_loss(y_true=labels_head1, y_pred=probs_head1, labels=[x for x in range(3)])
    acc_head1 = accuracy_score(y_true=labels_head1, y_pred=preds_head1.argmax(-1))
    
    probs_head2 = torch.from_numpy(preds_head2).float().softmax(-1).numpy()
    loss_head2 = log_loss(y_true=labels_head2, y_pred=probs_head2, labels=[x for x in range(58)])
    acc_head2 = accuracy_score(y_true=labels_head2, y_pred=preds_head2.argmax(-1))

    probs_head3 = torch.from_numpy(preds_head3).float().softmax(-1).numpy()
    loss_head3 = log_loss(y_true=labels_head3, y_pred=probs_head3, labels=[x for x in range(58)])
    acc_head3 = accuracy_score(y_true=labels_head3, y_pred=preds_head3.argmax(-1))
    
    # Return the metrics for each head
    return {
        "acc_classify": acc_head1,
        "log_loss_classify": loss_head1,
        "acc_model_a": acc_head2,
        "log_loss_model_a": loss_head2,
        "acc_model_b": acc_head3,
        "log_loss_model_b": loss_head3
    }

# Train Valid Split

Here, train and eval is splitted according to their `id % 5`

In [20]:
if TRAIN_100_PERCENT:
    folds = [
        (
            [i for i in range(len(ds))], 
            [i for i in range(len(ds)) if (i % config.n_splits == fold_idx)&(i<LN)]
        ) 
        for fold_idx in range(config.n_splits)
    ]
    print("We are training with 100% data")
else:
    folds = [
        (
            [i for i in range(len(ds)) if i % config.n_splits != fold_idx],
            [i for i in range(len(ds)) if (i % config.n_splits == fold_idx)&(i<LN)]
        ) 
        for fold_idx in range(config.n_splits)
    ]    

# Train Model

In [21]:
train_idx, eval_idx = folds[config.fold_idx]

trainer = Trainer(
    args=training_args, 
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds.select(train_idx),
    eval_dataset=ds.select(eval_idx),
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer.train()

/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Acc Classify,Log Loss Classify,Acc Model A,Log Loss Model A,Acc Model B,Log Loss Model B
0,No log,3.578812,0.307692,2.465684,0.000000,5.691593,0.000000,5.438289


TrainOutput(global_step=6, training_loss=3.0760377248128257, metrics={'train_runtime': 283.1085, 'train_samples_per_second': 0.18, 'train_steps_per_second': 0.021, 'total_flos': 2361348345882624.0, 'train_loss': 3.0760377248128257, 'epoch': 0.9230769230769231})

# Save LoRA Adapter and Heads

In [22]:
trainer.save_model(f"LoRA-v{VER}")